# Schemas and clinic configuration

This first demo validates the clinic's configuration, demonstrates schema validation errors, and persists the configuration in SQLite.

In [ ]:
$env:DB_PATH = "data/demo.sqlite"
$env:CHROMA_DIR = "chroma_db"
.\.venv\Scripts\python.exe scripts\seed_demo_data.py

In [ ]:
$env:DB_PATH = "data/demo.sqlite"
$env:CHROMA_DIR = "chroma_db"
.\.venv\Scripts\python.exe scripts\seed_demo_data.py

In [ ]:
$env:DB_PATH = "data/demo.sqlite"
$env:CHROMA_DIR = "chroma_db"
.\.venv\Scripts\python.exe scripts\seed_demo_data.py

In [ ]:
$env:DB_PATH = "data/demo.sqlite"
$env:CHROMA_DIR = "chroma_db"
.\.venv\Scripts\python.exe scripts\seed_demo_data.py

In [ ]:
$env:DB_PATH = "data/demo.sqlite"
$env:CHROMA_DIR = "chroma_db"
$env:STAFF_API_KEY = "<replace-with-a-long-random-value>"
$env:SEED_SAMPLE_CONFIG = "false"
.\.venv\Scripts\python.exe -m uvicorn app.api:app --reload

In [ ]:
import json
import sys
from datetime import date, datetime, time, timedelta
from pathlib import Path

from pydantic import ValidationError

project_root = Path.cwd()
if not (project_root / "app").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from app.db import load_clinic_config, save_clinic_config
from app.schemas import AppointmentRecord, BookingRequest, ClinicConfig

## Load and validate the clinic configuration

The sample JSON describes the clinic hours, policies, peak period, break, and services.

In [ ]:
config_path = project_root / "data" / "sample_clinic_config.json"
config = ClinicConfig.model_validate_json(config_path.read_text(encoding="utf-8"))
print(config)

## Invalid inputs are rejected

Pydantic reports invalid values rather than allowing inconsistent clinic hours or a booking date in the past.

In [ ]:
invalid_hours = json.loads(config_path.read_text(encoding="utf-8"))
invalid_hours["work_start"] = "21:00:00"
invalid_hours["work_end"] = "16:00:00"

try:
    ClinicConfig.model_validate(invalid_hours)
except ValidationError as exc:
    print("Invalid clinic hours:\n", exc)

In [ ]:
try:
    BookingRequest(
        patient_name="Demo Patient",
        patient_phone="+962-7-9000-0000",
        appointment_date=date.today(),
    )
except ValidationError as exc:
    print("Invalid booking date:\n", exc)

## Save and reload with SQLite

The database module uses SQLite directly and stores the validated configuration as JSON.

In [ ]:
save_clinic_config(config)
reloaded_config = load_clinic_config()
print("Configuration round trip successful:", reloaded_config == config)

## Example appointment record

This pending record includes the booking-time recommendation and fields intended for later outcome tracking.

In [ ]:
appointment_date = date.today() + timedelta(days=1)
example_appointment = AppointmentRecord(
    appointment_id=1,
    booking_timestamp=datetime.now(),
    appointment_date=appointment_date,
    day_of_week=appointment_date.strftime("%a"),
    scheduled_time=time(16, 0),
    queue_number=1,
    patients_before=0,
    bookings_that_day=1,
    is_peak=False,
    recommended_arrival=time(15, 55),
    predicted_wait_min=0,
    predicted_by="rules",
    outcome="pending",
)
print(example_appointment)